# 从完整 decoder 到可检查的训练

这个 Notebook 真正在 CPU 训练小模型。需要 PyTorch 2.8.0 CPU，命令见 README。语料是构造句子，loss 下降不代表泛化能力。

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/run_python.py").exists())
for p in ROOT.rglob("src"):
    if not {"node_modules", ".git", ".venv"}.intersection(p.parts):
        sys.path.insert(0, str(p))


In [2]:
import torch
from tiny_transformer.model import CharacterTokenizer, Config, Decoder, supervised_batch
torch.manual_seed(7);torch.set_num_threads(2)
tokenizer=CharacterTokenizer('hello answer')
model=Decoder(Config(len(tokenizer.vocabulary)))
x,y=supervised_batch(tokenizer,[('hello',' answer')])
logits,cache=model(x)
print({'input_ids':x.tolist(),'shifted_labels':y.tolist(),'logits_shape':list(logits.shape),'first_layer_key_shape':list(cache[0][0].shape)})

{'input_ids': [[1, 7, 6, 8, 8, 10, 4, 5, 9, 12, 13, 6, 11]], 'shifted_labels': [[-100, -100, -100, -100, -100, 4, 5, 9, 12, 13, 6, 11, 2]], 'logits_shape': [1, 13, 14], 'first_layer_key_shape': [1, 4, 13, 8]}


## 训练、LoRA、缓存和重载

所有结果来自本次运行。检查冻结权重没有更新，再看 loss；只看 loss 会漏掉误解冻问题。

In [3]:
from tiny_transformer.experiment import experiment
with tempfile.TemporaryDirectory() as directory:
    report=experiment(directory,steps=65)
assert report['pretraining_loss'][1]<report['pretraining_loss'][0]
assert report['frozen_weights_unchanged']
print(json.dumps(report,ensure_ascii=False,indent=2))

{
  "seed": 7,
  "device": "cpu",
  "torch": "2.8.0+cpu",
  "parameters": 31464,
  "pretraining_loss": [
    3.253199338912964,
    0.0016245456645265222
  ],
  "sft_loss": [
    3.8780176639556885,
    0.003843255341053009
  ],
  "sft_trainable_parameters": [
    {
      "name": "head.a",
      "shape": [
        4,
        32
      ],
      "numel": 128
    },
    {
      "name": "head.b",
      "shape": [
        26,
        4
      ],
      "numel": 104
    }
  ],
  "sft_prediction_ids_before": [
    [
      22,
      18,
      18,
      23,
      15,
      14,
      5,
      4,
      22,
      15,
      5,
      24,
      24,
      15,
      6,
      6,
      5,
      22
    ],
    [
      22,
      18,
      18,
      23,
      18,
      15,
      11,
      19,
      23,
      5,
      23,
      24,
      5,
      18,
      6,
      21,
      5,
      22
    ]
  ],
  "sft_prediction_ids_after": [
    [
      2,
      16,
      5,
      5,
      6,
      6,
      4,
      4,
     

## 修改一个条件再观察

在源码中查看 Attention.forward 的 prefix 和 mask。缓存时位置必须加上旧前缀长度。先理解为什么，再在副本中故意删除偏移，运行 tests/test_decoder.py，观察缓存一致性测试失败。不要把故意破坏的副本当作正确实现提交。